# LeetCode 3321 — Find X-Sum of All K-Long Subarrays II bằng Python

- Link: https://leetcode.com/problems/find-x-sum-of-all-k-long-subarrays-ii/
- Độ khó: **Hard**
- Chủ đề: **Cửa sổ trượt** + duy trì **top-x** theo khóa `(tần suất, giá trị)` bằng **hai tập có thứ tự** (SortedList) và tổng chạy; cách khác: **Fenwick tree offline** trên khóa nén

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findXSum(self, nums: List[int], k: int, x: int) -> List[int]:
```

## 1. Mục tiêu học

1. Hiểu x-sum: giữ lại `x` giá trị "phổ biến nhất" theo khóa `(tần suất, giá trị)` (tần suất lớn hơn trước; hòa thì **giá trị lớn hơn** trước), rồi cộng `tần suất · giá trị`.
2. Khi cửa sổ trượt, chỉ **hai** khóa thay đổi (phần tử vào, phần tử ra) ⇒ cập nhật cấu trúc dữ liệu `O(log n)`.
3. Mẫu **hai tập** `TOP` (x khóa lớn nhất) và `REST`, cùng tổng `Σ f·v` của `TOP`; sau mỗi thay đổi **cân bằng lại**.
4. Cách thay thế không cần SortedList: **nén offline** mọi khóa `(f, v)` có thể xuất hiện, dùng Fenwick tree đếm/tổng và **tìm phần tử thứ `t`** bằng nhảy nhị phân.
5. Kiểm chứng bằng vét cạn từng cửa sổ.

## 2. Đề bài & ví dụ

Cho `nums`, `k`, `x`. **x-sum** của một mảng: đếm tần suất mọi phần tử; chỉ giữ các lần xuất hiện của `x` phần tử phổ biến nhất (hòa tần suất thì phần tử **lớn hơn** được coi là phổ biến hơn); trả về tổng mảng còn lại. Nếu mảng có ít hơn `x` phần tử phân biệt, x-sum là tổng cả mảng. Trả về x-sum của mọi đoạn con độ dài `k`.

**Ví dụ 1:** `nums = [1,1,2,2,3,4,2,3]`, `k = 6`, `x = 2` → `[6,10,12]`.
**Ví dụ 2:** `nums = [3,8,7,8,7,5]`, `k = 2`, `x = 2` → `[11,15,15,15,12]`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`; `1 ≤ nums[i] ≤ 10^9`; `1 ≤ x ≤ k ≤ n`.

In [1]:
from typing import List
from collections import Counter, defaultdict
import bisect

examples = [([1, 1, 2, 2, 3, 4, 2, 3], 6, 2, [6, 10, 12]), ([3, 8, 7, 8, 7, 5], 2, 2, [11, 15, 15, 15, 12])]
for nums, k, x, exp in examples:
    print(nums, 'k =', k, 'x =', x, '-> kỳ vọng', exp)

[1, 1, 2, 2, 3, 4, 2, 3] k = 6 x = 2 -> kỳ vọng [6, 10, 12]
[3, 8, 7, 8, 7, 5] k = 2 x = 2 -> kỳ vọng [11, 15, 15, 15, 12]


## 3. Kiến thức nền

### 3.1. x-sum của một mảng

In [2]:
def x_sum(arr, x):
    cnt = Counter(arr)
    top = sorted(cnt.items(), key=lambda p: (p[1], p[0]), reverse=True)[:x]
    return sum(v * f for v, f in top)

print(x_sum([1, 2, 2, 3, 4, 2], 2), '(giữ 2 ×3 và 4 ×1)')

10 (giữ 2 ×3 và 4 ×1)


### 3.2. SortedList

LeetCode có sẵn thư viện `sortedcontainers` (`SortedList`: `add`, `remove`, truy cập `[0]`, `[-1]` trong `O(log n)`). Để notebook chạy được ở mọi máy, ta dùng nó nếu có, nếu không thì dùng một bản **danh sách phân khối** tối giản (mỗi khối ≤ 512 phần tử, thao tác `O(√n)`).

In [3]:
try:
    from sortedcontainers import SortedList
except ImportError:
    class SortedList:
        """Bản tối giản: các khối đã sắp, mỗi khối tối đa LOAD phần tử."""
        LOAD = 512
        def __init__(self, it=()):
            a = sorted(it)
            self.b = [a[i:i + self.LOAD] for i in range(0, len(a), self.LOAD)]
            self.n = len(a)
        def __len__(self):
            return self.n
        def _find(self, v):
            for i, blk in enumerate(self.b):
                if v <= blk[-1]:
                    return i
            return len(self.b) - 1
        def add(self, v):
            if not self.b:
                self.b = [[v]]
            else:
                i = self._find(v)
                bisect.insort(self.b[i], v)
                if len(self.b[i]) > 2 * self.LOAD:
                    blk = self.b[i]
                    self.b[i:i + 1] = [blk[:self.LOAD], blk[self.LOAD:]]
            self.n += 1
        def remove(self, v):
            i = self._find(v)
            blk = self.b[i]
            j = bisect.bisect_left(blk, v)
            if j == len(blk) or blk[j] != v:
                raise ValueError(v)
            blk.pop(j)
            if not blk:
                self.b.pop(i)
            self.n -= 1
        def __getitem__(self, idx):
            if idx == 0:
                return self.b[0][0]
            if idx == -1:
                return self.b[-1][-1]
            if idx < 0:
                idx += self.n
            for blk in self.b:
                if idx < len(blk):
                    return blk[idx]
                idx -= len(blk)
            raise IndexError
        def __iter__(self):
            for blk in self.b:
                yield from blk

sl = SortedList([5, 1, 3])
sl.add(4); sl.remove(1)
print(list(sl), sl[0], sl[-1], len(sl))

[3, 4, 5] 3 5 3


### 3.3. Khóa so sánh

Dùng tuple `(f, v)`: so sánh tuple của Python ưu tiên `f`, hòa thì so `v` — đúng thứ tự "phổ biến hơn" của đề (lớn hơn = phổ biến hơn).

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1, cửa sổ 1 `[1,1,2,2,3,4]` → cửa sổ 2 `[1,2,2,3,4,2]`: ra một `1`, vào một `2`.

- Khóa `(2, 1)` → `(1, 1)`; khóa `(2, 2)` → `(3, 2)`.
- Thứ tự khóa: `(3,2) > (1,4) > (1,3) > (1,1)` ⇒ top-2 là `2` (×3) và `4` (×1) ⇒ `6 + 4 = 10` ✓.

Chỉ hai khóa đổi mỗi bước ⇒ không cần sắp lại toàn bộ.

## 5. Cách 1 — Brute Force: tính x-sum cho từng cửa sổ

In [4]:
def brute(nums, k, x):
    return [x_sum(nums[i:i + k], x) for i in range(len(nums) - k + 1)]

for nums, k, x, exp in examples:
    print(brute(nums, k, x), '| kỳ vọng', exp)

[6, 10, 12] | kỳ vọng [6, 10, 12]
[11, 15, 15, 15, 12] | kỳ vọng [11, 15, 15, 15, 12]


**Độ phức tạp:** `O((n − k + 1) · k log k)`.

## 6. Ý tưởng thuật toán / bước đột phá

Giữ `cnt[v]`, hai `SortedList` `TOP`, `REST` chứa khóa `(cnt[v], v)` của các `v` đang có mặt, và `S = Σ_{(f, v) ∈ TOP} f · v`.

- `erase(v)`: xóa khóa hiện tại `(cnt[v], v)` khỏi tập đang chứa nó (trừ `S` nếu ở `TOP`).
- `insert(v)`: thêm khóa `(cnt[v], v)`: nếu lớn hơn `TOP[0]` thì vào `TOP` (cộng `S`), ngược lại vào `REST`.
- `rebalance()`: nếu `|TOP| > x` chuyển `TOP[0]` xuống `REST`; nếu `|TOP| < x` và `REST` khác rỗng chuyển `REST[−1]` lên.

Thêm/bớt một phần tử = `erase`, đổi `cnt`, `insert` (nếu `cnt > 0`), `rebalance`. Mỗi bước `O(log n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đếm trượt + sắp các giá trị phân biệt mỗi cửa sổ

In [5]:
def sliding_sort(nums, k, x):
    cnt = Counter(nums[:k])
    out = []
    for i in range(len(nums) - k + 1):
        if i:
            cnt[nums[i - 1]] -= 1
            if cnt[nums[i - 1]] == 0:
                del cnt[nums[i - 1]]
            cnt[nums[i + k - 1]] += 1
        top = sorted(((f, v) for v, f in cnt.items()), reverse=True)[:x]
        out.append(sum(f * v for f, v in top))
    return out

for nums, k, x, exp in examples:
    print(sliding_sort(nums, k, x), '| kỳ vọng', exp)

[6, 10, 12] | kỳ vọng [6, 10, 12]
[11, 15, 15, 15, 12] | kỳ vọng [11, 15, 15, 15, 12]


`O(n · d log d)` với `d` = số giá trị phân biệt trong cửa sổ — tốt khi `d` nhỏ.

### 7.2. Cách 3 — Hai SortedList + tổng chạy (lời giải chính)

In [6]:
def two_sets(nums, k, x):
    cnt = defaultdict(int)
    TOP, REST = SortedList(), SortedList()
    S = 0

    def erase(v):
        nonlocal S
        key = (cnt[v], v)
        if TOP and key >= TOP[0]:
            TOP.remove(key); S -= key[0] * key[1]
        else:
            REST.remove(key)

    def insert(v):
        nonlocal S
        key = (cnt[v], v)
        if TOP and key > TOP[0]:
            TOP.add(key); S += key[0] * key[1]
        else:
            REST.add(key)

    def rebalance():
        nonlocal S
        while len(TOP) > x:
            key = TOP[0]; TOP.remove(key); S -= key[0] * key[1]; REST.add(key)
        while len(TOP) < x and REST:
            key = REST[-1]; REST.remove(key); TOP.add(key); S += key[0] * key[1]

    def change(v, d):
        if cnt[v]:
            erase(v)
        cnt[v] += d
        if cnt[v]:
            insert(v)
        rebalance()

    out = []
    for i, v in enumerate(nums):
        change(v, +1)
        if i >= k:
            change(nums[i - k], -1)
        if i >= k - 1:
            out.append(S)
    return out

for nums, k, x, exp in examples:
    print(two_sets(nums, k, x), '| kỳ vọng', exp)

[6, 10, 12] | kỳ vọng [6, 10, 12]
[11, 15, 15, 15, 12] | kỳ vọng [11, 15, 15, 15, 12]


Bất biến sau mỗi `rebalance`: `|TOP| = min(x, số khóa)` và mọi khóa trong `TOP` lớn hơn mọi khóa trong `REST`. Chú ý khi `insert` vào `REST` một khóa lớn hơn `REST[−1]` nhưng nhỏ hơn `TOP[0]` — vẫn đúng thứ tự.

### 7.3. Cách 4 — Fenwick tree offline trên khóa nén

Mọi khóa có thể xuất hiện là `(f, v)` sau mỗi lần thay đổi ⇒ mô phỏng trước để thu thập (≤ `2n` khóa), nén thứ tự. Fenwick lưu **số khóa** và **tổng `f·v`** theo thứ hạng. Top-x = các khóa có hạng cao nhất: tìm hạng của khóa thứ `(C − x)` (C = số khóa đang có) bằng nhảy nhị phân, đáp án = tổng toàn bộ − tổng tiền tố tới đó.

In [7]:
def fenwick_offline(nums, k, x):
    # 1) thu thập mọi khóa
    cnt = defaultdict(int); keys = set()
    def touch(v, d):
        cnt[v] += d
        if cnt[v]:
            keys.add((cnt[v], v))
    for i, v in enumerate(nums):
        touch(v, 1)
        if i >= k:
            touch(nums[i - k], -1)
    order = sorted(keys)
    rank = {key: i + 1 for i, key in enumerate(order)}
    m = len(order)
    LOG = m.bit_length()
    fc, fs = [0] * (m + 1), [0] * (m + 1)

    def upd(i, dc, ds):
        while i <= m:
            fc[i] += dc; fs[i] += ds; i += i & -i

    def prefix_by_count(t):        # tổng f·v của t khóa nhỏ nhất
        pos = 0; s = 0
        for b in range(LOG, -1, -1):
            nxt = pos + (1 << b)
            if nxt <= m and fc[nxt] <= t:
                pos = nxt; t -= fc[nxt]; s += fs[nxt]
        return s

    cnt = defaultdict(int)
    C = total = 0
    out = []
    def change(v, d):
        nonlocal C, total
        if cnt[v]:
            key = (cnt[v], v); upd(rank[key], -1, -key[0] * key[1]); C -= 1; total -= key[0] * key[1]
        cnt[v] += d
        if cnt[v]:
            key = (cnt[v], v); upd(rank[key], 1, key[0] * key[1]); C += 1; total += key[0] * key[1]
    for i, v in enumerate(nums):
        change(v, 1)
        if i >= k:
            change(nums[i - k], -1)
        if i >= k - 1:
            out.append(total - (prefix_by_count(C - x) if C > x else 0))
    return out

for nums, k, x, exp in examples:
    print(fenwick_offline(nums, k, x), '| kỳ vọng', exp)

[6, 10, 12] | kỳ vọng [6, 10, 12]
[11, 15, 15, 15, 12] | kỳ vọng [11, 15, 15, 15, 12]


Vì mỗi khóa đang hoạt động xuất hiện **một lần** (mỗi giá trị chỉ có một tần suất tại một thời điểm), Fenwick đếm số khóa chính là đếm số giá trị phân biệt.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1, in `TOP`, `REST`, `S` sau mỗi cửa sổ.

In [8]:
def trace(nums, k, x):
    cnt = defaultdict(int)
    for i in range(len(nums) - k + 1):
        window = nums[i:i + k]
        keys = sorted(((f, v) for v, f in Counter(window).items()), reverse=True)
        top, rest = keys[:x], keys[x:]
        print(f"cửa sổ {window}: TOP = {top}, REST = {rest} -> S = {sum(f * v for f, v in top)}")
    print('cách 3:', two_sets(nums, k, x), '| vét cạn:', brute(nums, k, x))

trace(*examples[0][:3])

cửa sổ [1, 1, 2, 2, 3, 4]: TOP = [(2, 2), (2, 1)], REST = [(1, 4), (1, 3)] -> S = 6
cửa sổ [1, 2, 2, 3, 4, 2]: TOP = [(3, 2), (1, 4)], REST = [(1, 3), (1, 1)] -> S = 10
cửa sổ [2, 2, 3, 4, 2, 3]: TOP = [(3, 2), (2, 3)], REST = [(1, 4)] -> S = 12
cách 3: [6, 10, 12] | vét cạn: [6, 10, 12]


## 9. Code nộp LeetCode

Trên LeetCode `sortedcontainers` có sẵn; trong notebook `SortedList` đã được chuẩn bị ở mục 3.2.

In [9]:
class Solution:
    def findXSum(self, nums: List[int], k: int, x: int) -> List[int]:
        # from sortedcontainers import SortedList   # có sẵn trên LeetCode
        cnt = defaultdict(int)
        TOP, REST = SortedList(), SortedList()
        S = 0
        ans = []

        def change(v, d):
            nonlocal S
            f = cnt[v]
            if f:
                key = (f, v)
                if TOP and key >= TOP[0]:
                    TOP.remove(key); S -= f * v
                else:
                    REST.remove(key)
            f += d
            cnt[v] = f
            if f:
                key = (f, v)
                if TOP and key > TOP[0]:
                    TOP.add(key); S += f * v
                else:
                    REST.add(key)
            while len(TOP) > x:
                key = TOP[0]; TOP.remove(key); S -= key[0] * key[1]; REST.add(key)
            while len(TOP) < x and REST:
                key = REST[-1]; REST.remove(key); TOP.add(key); S += key[0] * key[1]

        for i, v in enumerate(nums):
            change(v, 1)
            if i >= k:
                change(nums[i - k], -1)
            if i >= k - 1:
                ans.append(S)
        return ans

for nums, k, x, exp in examples:
    got = Solution().findXSum(nums, k, x)
    print(got, 'OK' if got == exp else 'SAI')

[6, 10, 12] OK
[11, 15, 15, 15, 12] OK


## 10. Vì sao thuật toán đúng

**Bất biến** (sau mỗi `change`): (1) `TOP ∪ REST` = tập khóa `(cnt[v], v)` của các `v` có `cnt[v] > 0`; (2) `|TOP| = min(x, |TOP ∪ REST|)`; (3) mọi khóa của `TOP` lớn hơn mọi khóa của `REST`; (4) `S = Σ_{TOP} f·v`.

- `erase`: theo (3), khóa nằm ở `TOP` ⇔ nó `≥ TOP[0]`.
- `insert`: đặt khóa vào phía giữ được (3) (khóa lớn hơn `TOP[0]` vào `TOP`, ngược lại vào `REST` — vẫn nhỏ hơn mọi khóa `TOP`).
- `rebalance`: chuyển phần tử biên (`TOP[0]` là nhỏ nhất của `TOP`, `REST[−1]` lớn nhất của `REST`) không phá (3), và khôi phục (2).

Do các khóa phân biệt (mỗi `v` một khóa), `TOP` đúng là `x` giá trị phổ biến nhất theo luật đề ⇒ `S` là x-sum.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n k log k)` | `O(k)` |
| 2. Đếm trượt + sắp | `O(n d log d)` | `O(d)` |
| 3. Hai SortedList | `O(n log n)` | `O(n)` |
| 4. Fenwick offline | `O(n log n)` | `O(n)` |

Với bản `SortedList` tối giản (khi không có `sortedcontainers`), mỗi thao tác `O(√n)` — vẫn đủ nhanh cho `n = 10^5`.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'sliding_sort': sliding_sort, 'two_sets': two_sets, 'fenwick': fenwick_offline,
         'submit': lambda a, k, x: Solution().findXSum(a, k, x)}
edge = [
    ([5], 1, 1, [5]),
    ([1, 2, 3], 3, 1, [3]),                 # hòa tần suất -> giá trị lớn nhất
    ([1, 2, 3], 3, 5, [6]),                 # ít hơn x giá trị phân biệt
    ([2, 2, 1, 1, 1], 4, 1, [4, 3]),
    ([7, 7, 7, 7], 2, 1, [14, 14, 14]),
]
for nums, k, x, exp in examples + edge:
    assert brute(nums, k, x) == exp, (nums, k, x, brute(nums, k, x))
    for name, f in impls.items():
        assert f(nums, k, x) == exp, (name, nums, k, x)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3321)
cnt_tests = 0
for _ in range(500):
    n = rng.randint(1, 14)
    nums = [rng.randint(1, rng.choice([3, 6, 50])) for _ in range(n)]
    k = rng.randint(1, n); x = rng.randint(1, k)
    exp = brute(nums, k, x)
    for name, f in impls.items():
        assert f(nums, k, x) == exp, (name, nums, k, x)
    cnt_tests += 1
print('Khớp vét cạn trên', cnt_tests, 'bộ test.')

Khớp vét cạn trên 500 bộ test.


### 12.3. Ca lớn `n = 10^5`

In [12]:
import time
N = 10 ** 5
big = {
    'giá trị 1..1000, k = 5e4, x = 100': ([rng.randint(1, 1000) for _ in range(N)], 5 * 10 ** 4, 100),
    'giá trị tới 1e9, k = 1000, x = 10': ([rng.randint(1, 10 ** 9) for _ in range(N)], 1000, 10),
    'ít giá trị, k = N, x = 1': ([rng.randint(1, 5) for _ in range(N)], N, 1),
}
for name, (nums, k, x) in big.items():
    t = time.time(); a = Solution().findXSum(nums, k, x); t1 = time.time() - t
    t = time.time(); b = fenwick_offline(nums, k, x); t2 = time.time() - t
    assert a == b
    print(f"{name:36s}: {len(a):6d} cửa sổ | nộp {t1:.2f}s | Fenwick {t2:.2f}s")
nums, k, x = big['ít giá trị, k = N, x = 1']
assert Solution().findXSum(nums, k, x) == [x_sum(nums, 1)]
print('k = N: khớp x_sum trực tiếp.')

giá trị 1..1000, k = 5e4, x = 100   :  50001 cửa sổ | nộp 0.36s | Fenwick 0.64s


giá trị tới 1e9, k = 1000, x = 10   :  99001 cửa sổ | nộp 0.26s | Fenwick 0.82s


ít giá trị, k = N, x = 1            :      1 cửa sổ | nộp 0.15s | Fenwick 0.49s
k = N: khớp x_sum trực tiếp.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Hòa tần suất thì ưu tiên giá trị **nhỏ** hơn

In [13]:
def wrong_tie(nums, k, x):
    out = []
    for i in range(len(nums) - k + 1):
        cnt = Counter(nums[i:i + k])
        top = sorted(cnt.items(), key=lambda p: (-p[1], p[0]))[:x]    # SAI: hòa thì lấy giá trị nhỏ
        out.append(sum(v * f for v, f in top))
    return out

print('sai :', wrong_tie(*examples[0][:3]), '| đúng:', Solution().findXSum(*examples[0][:3]))

sai : [6, 7, 12] | đúng: [6, 10, 12]


**Vì sao sai:** cửa sổ 2 có `1, 3, 4` cùng tần suất 1; đề chọn `4` (giá trị lớn hơn).

### Lỗi 2 — Cộng mỗi giá trị **một lần** thay vì `tần suất · giá trị`

In [14]:
def wrong_once(nums, k, x):
    out = []
    for i in range(len(nums) - k + 1):
        cnt = Counter(nums[i:i + k])
        top = sorted(cnt.items(), key=lambda p: (p[1], p[0]), reverse=True)[:x]
        out.append(sum(v for v, f in top))                               # SAI
    return out

print('sai :', wrong_once(*examples[0][:3]), '| đúng:', Solution().findXSum(*examples[0][:3]))

sai : [3, 6, 5] | đúng: [6, 10, 12]


**Vì sao sai:** đề giữ **các lần xuất hiện** của các phần tử được chọn rồi cộng — `1 + 1 + 2 + 2 = 6`, không phải `1 + 2`.

### Lỗi 3 — Không xóa khóa cũ trước khi thêm khóa mới

In [15]:
def wrong_stale(nums, k, x):
    cnt = defaultdict(int)
    TOP, REST = SortedList(), SortedList()
    S = 0
    out = []
    def change(v, d):
        nonlocal S
        cnt[v] += d                                  # SAI: khóa (cnt cũ, v) vẫn nằm trong tập
        if cnt[v]:
            key = (cnt[v], v)
            if TOP and key > TOP[0]:
                TOP.add(key); S += key[0] * key[1]
            else:
                REST.add(key)
        while len(TOP) > x:
            key = TOP[0]; TOP.remove(key); S -= key[0] * key[1]; REST.add(key)
        while len(TOP) < x and REST:
            key = REST[-1]; REST.remove(key); TOP.add(key); S += key[0] * key[1]
    for i, v in enumerate(nums):
        change(v, 1)
        if i >= k:
            change(nums[i - k], -1)
        if i >= k - 1:
            out.append(S)
    return out

bad = ([2, 1], 1, 1)
print('sai :', wrong_stale(*bad), '| đúng:', Solution().findXSum(*bad))

sai : [2, 2] | đúng: [2, 1]


**Vì sao sai:** với `[2, 1]`, `k = 1`: khi `2` rời cửa sổ, khóa `(1, 2)` không bị xóa, vẫn đứng đầu `TOP` và được tính vào `S` ⇒ cửa sổ `[1]` cho `2` thay vì `1`. Tương tự, khi tần suất tăng, khóa cũ và mới cùng tồn tại làm giá trị bị đếm hai lần.

### Lỗi 4 — Khi bỏ phần tử, kiểm tra thuộc `TOP` bằng `key > TOP[0]` (thiếu dấu bằng)

In [16]:
def wrong_membership(nums, k, x):
    cnt = defaultdict(int)
    TOP, REST = SortedList(), SortedList()
    S = 0
    out = []
    def change(v, d):
        nonlocal S
        f = cnt[v]
        if f:
            key = (f, v)
            if TOP and key > TOP[0]:                 # SAI: chính TOP[0] bị coi là ở REST
                TOP.remove(key); S -= f * v
            else:
                REST.remove(key) if key in list(REST) else None
        cnt[v] = f + d
        if cnt[v]:
            key = (cnt[v], v)
            if TOP and key > TOP[0]:
                TOP.add(key); S += key[0] * key[1]
            else:
                REST.add(key)
        while len(TOP) > x:
            key = TOP[0]; TOP.remove(key); S -= key[0] * key[1]; REST.add(key)
        while len(TOP) < x and REST:
            key = REST[-1]; REST.remove(key); TOP.add(key); S += key[0] * key[1]
    for i, v in enumerate(nums):
        change(v, 1)
        if i >= k:
            change(nums[i - k], -1)
        if i >= k - 1:
            out.append(S)
    return out

print('sai :', wrong_membership(*examples[1][:3]), '| đúng:', Solution().findXSum(*examples[1][:3]))

sai : [11, 15, 15, 15, 14] | đúng: [11, 15, 15, 15, 12]


**Vì sao sai:** khóa nhỏ nhất của `TOP` **bằng** `TOP[0]`, nên điều kiện phải là `>=`. Bản sai không xóa nó khỏi `TOP` (khóa cũ "mắc kẹt" và vẫn được tính trong `S`). Trong bản chuẩn, `REST.remove` sẽ báo lỗi ngay nếu nhầm — ở đây ta bỏ qua để thấy đáp án lệch.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Top-x theo một tiêu chí động trong cửa sổ trượt" ⇒ hai tập có thứ tự (hoặc hai heap với xóa trễ) + tổng chạy — cùng họ với "trung vị cửa sổ trượt" (LC 480).
- Khóa nhiều tiêu chí ⇒ tuple; nhớ chiều ưu tiên khi hòa.
- Không có SortedList ⇒ nén offline + Fenwick (tìm phần tử thứ `t` bằng nhảy nhị phân).

## 15. Biến thể của bài

1. **LC 3318 (bản I):** `n ≤ 50` — vét cạn.
2. Trả về **danh sách các giá trị** được chọn ở mỗi cửa sổ.
3. Hòa tần suất thì ưu tiên **giá trị nhỏ** ⇒ khóa `(f, −v)`.
4. Trung vị cửa sổ trượt (LC 480) — cùng kỹ thuật hai tập.

In [17]:
# Biến thể 3 (hòa -> giá trị nhỏ) và 4 (trung vị cửa sổ, k lẻ)
def x_sum_small_tie(nums, k, x):
    out = []
    for i in range(len(nums) - k + 1):
        cnt = Counter(nums[i:i + k])
        top = sorted(((f, -v) for v, f in cnt.items()), reverse=True)[:x]
        out.append(sum(f * -nv for f, nv in top))
    return out

def sliding_median_odd(nums, k):
    lo, hi = SortedList(), SortedList()         # lo: nửa nhỏ (chứa trung vị), hi: nửa lớn
    out = []
    def balance():
        while len(lo) > len(hi) + 1:
            v = lo[-1]; lo.remove(v); hi.add(v)
        while len(lo) < len(hi) + 1 and hi:
            v = hi[0]; hi.remove(v); lo.add(v)
    for i, v in enumerate(nums):
        key = (v, i)
        (lo.add(key) if not lo or key <= lo[-1] else hi.add(key)); balance()
        if i >= k:
            old = (nums[i - k], i - k)
            (lo.remove(old) if old <= lo[-1] else hi.remove(old)); balance()
        if i >= k - 1:
            out.append(lo[-1][0])
    return out

assert x_sum_small_tie([1, 1, 2, 2, 3, 4, 2, 3], 6, 2) == [6, 7, 12]
rng = random.Random(15)
for _ in range(300):
    nums = [rng.randint(1, 9) for _ in range(rng.randint(1, 12))]
    k = rng.randrange(1, len(nums) + 1, 2) if len(nums) >= 1 else 1
    assert sliding_median_odd(nums, k) == [sorted(nums[i:i + k])[k // 2] for i in range(len(nums) - k + 1)]
print('Biến thể 3 đúng ví dụ; biến thể 4 (trung vị cửa sổ) khớp vét cạn.')

Biến thể 3 đúng ví dụ; biến thể 4 (trung vị cửa sổ) khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | x-sum từng cửa sổ | `O(n k log k)` | Oracle |
| 2 | Đếm trượt + sắp | `O(n d log d)` | `d` nhỏ |
| 3 | Hai SortedList | `O(n log n)` | **Nộp bài** |
| 4 | Fenwick offline | `O(n log n)` | Không cần thư viện |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [4, 4, 4]`, `k = 2`, `x = 1`.

**Bài 2 (tính tay):** `nums = [1, 2, 1, 3]`, `k = 3`, `x = 1`.

**Bài 3 (điền TODO):** trả về danh sách các giá trị được giữ ở mỗi cửa sổ (biến thể 2).

**Bài 4 (tự viết lại):** x-sum cho mọi **đoạn tiền tố** `nums[0..i]`.

**Bài 5 (mini challenge):** cài đặt cách 3 bằng **hai heap + xóa trễ** (không dùng SortedList).

In [18]:
def chosen_values_todo(nums, k, x):
    # TODO
    raise NotImplementedError

def prefix_x_sums_todo(nums, x):
    # TODO
    raise NotImplementedError

def two_heaps_todo(nums, k, x):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Mỗi cửa sổ `[4, 4]`. (2) Một giá trị. (3) `[8, 8]`.

**Bài 2.** (1) `[1,2,1]`: `1` ×2 ⇒ 2. (2) `[2,1,3]`: hòa ⇒ chọn `3`. (3) `[2, 3]`.

**Bài 3.** (1) Giữ `TOP` như cách 3. (2) Xuất `sorted(v for f, v in TOP)`. (3) So vét cạn.

**Bài 4.** (1) Chỉ thêm, không bớt. (2) Dùng `change(v, +1)` của cách 3. (3) Xuất `S` sau mỗi bước.

**Bài 5.** (1) `TOP` là min-heap, `REST` là max-heap (khóa âm). (2) Lưu "phiên bản" khóa hiện tại của mỗi `v`; khi đỉnh heap là khóa cũ thì bỏ. (3) Đếm kích thước **hợp lệ** riêng thay vì `len(heap)`.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3, 4
assert Solution().findXSum([4, 4, 4], 2, 1) == [8, 8]
assert Solution().findXSum([1, 2, 1, 3], 3, 1) == [2, 3]

def chosen_values(nums, k, x):
    res = []
    for i in range(len(nums) - k + 1):
        cnt = Counter(nums[i:i + k])
        res.append(sorted(v for v, f in sorted(cnt.items(), key=lambda p: (p[1], p[0]), reverse=True)[:x]))
    return res

def prefix_x_sums(nums, x):
    cnt = defaultdict(int); TOP, REST = SortedList(), SortedList(); S = 0; out = []
    for v in nums:
        f = cnt[v]
        if f:
            key = (f, v)
            if TOP and key >= TOP[0]:
                TOP.remove(key); S -= f * v
            else:
                REST.remove(key)
        cnt[v] = f + 1; key = (f + 1, v)
        if TOP and key > TOP[0]:
            TOP.add(key); S += key[0] * key[1]
        else:
            REST.add(key)
        while len(TOP) > x:
            kk = TOP[0]; TOP.remove(kk); S -= kk[0] * kk[1]; REST.add(kk)
        while len(TOP) < x and REST:
            kk = REST[-1]; REST.remove(kk); TOP.add(kk); S += kk[0] * kk[1]
        out.append(S)
    return out

rng = random.Random(3)
for _ in range(200):
    nums = [rng.randint(1, 6) for _ in range(rng.randint(1, 10))]; x = rng.randint(1, 4)
    assert prefix_x_sums(nums, x) == [x_sum(nums[:i + 1], x) for i in range(len(nums))]
print('Bài 1: [8, 8] | Bài 2: [2, 3] | Bài 3:', chosen_values(*examples[0][:3]), '| Bài 4 khớp vét cạn.')

Bài 1: [8, 8] | Bài 2: [2, 3] | Bài 3: [[1, 2], [2, 4], [2, 3]] | Bài 4 khớp vét cạn.


In [20]:
# Bài 5: hai heap + xóa trễ
import heapq
def two_heaps(nums, k, x):
    cnt = defaultdict(int)
    where = {}                         # v -> 'T' / 'R' cho khóa hiện tại (cnt[v], v)
    top, rest = [], []                 # top: min-heap (f, v); rest: max-heap (-f, -v)
    size_top = 0; S = 0
    def valid(f, v):
        return cnt[v] == f and f > 0
    def clean():
        while top and not (valid(*top[0]) and where.get(top[0][1]) == 'T'):
            heapq.heappop(top)
        while rest and not (valid(-rest[0][0], -rest[0][1]) and where.get(-rest[0][1]) == 'R'):
            heapq.heappop(rest)
    def push_rest(v):
        where[v] = 'R'; heapq.heappush(rest, (-cnt[v], -v))
    def push_top(v):
        nonlocal size_top, S
        where[v] = 'T'; heapq.heappush(top, (cnt[v], v)); size_top += 1; S += cnt[v] * v
    def change(v, d):
        nonlocal size_top, S
        if cnt[v] and where.get(v) == 'T':
            size_top -= 1; S -= cnt[v] * v
        where.pop(v, None)
        cnt[v] += d
        if cnt[v]:
            push_rest(v)
        while True:                       # cân bằng: đưa phần tử tốt nhất của rest lên nếu cần
            clean()
            if rest and (size_top < x or (top and (-rest[0][0], -rest[0][1]) > top[0])):
                f, v2 = -rest[0][0], -rest[0][1]; heapq.heappop(rest); push_top(v2)
                clean()
                if size_top > x:
                    f3, v3 = heapq.heappop(top); size_top -= 1; S -= f3 * v3; push_rest(v3)
            else:
                break
    out = []
    for i, v in enumerate(nums):
        change(v, 1)
        if i >= k:
            change(nums[i - k], -1)
        if i >= k - 1:
            out.append(S)
    return out

rng = random.Random(4)
for _ in range(400):
    n = rng.randint(1, 12); nums = [rng.randint(1, 6) for _ in range(n)]
    k = rng.randint(1, n); x = rng.randint(1, k)
    assert two_heaps(nums, k, x) == brute(nums, k, x), (nums, k, x)
print('Bài 5: hai heap + xóa trễ khớp vét cạn.')

Bài 5: hai heap + xóa trễ khớp vét cạn.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
change(v, d):
    xóa khóa (cnt[v], v) khỏi TOP (nếu >= TOP[0]) hoặc REST
    cnt[v] += d; nếu > 0: thêm vào TOP (nếu > TOP[0]) hoặc REST
    while |TOP| > x: TOP[0] -> REST;  while |TOP| < x and REST: REST[-1] -> TOP
S = Σ f·v của TOP  (cập nhật mỗi lần di chuyển)
```

**Checklist:**

- [ ] Khóa `(f, v)` — hòa tần suất thì giá trị **lớn** hơn thắng?
- [ ] Cộng `f · v`, không phải `v`?
- [ ] Xóa khóa cũ trước khi thêm khóa mới?
- [ ] Điều kiện thuộc `TOP` là `>= TOP[0]`?
- [ ] Ít hơn `x` giá trị phân biệt ⇒ `TOP` chứa hết?